# Pulse calibration workflow

This notebook calibrates the default half-pi / pi pulses first, then
shows how to refine the DRAG versions. The goal is to leave you with
inspectable cached pulses and quick validation plots.

## 1. Create an `Experiment`

Select one qubit and load the configuration that contains its current
pulse parameters.

In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

In [ ]:
Q0 = exp.qubit_labels[0]

print("target qubit:", Q0)

## 2. Connect and obtain a baseline Rabi fit

The Rabi fit gives you a starting point for amplitude-based pulse
calibration.

In [ ]:
exp.connect()

rabi_result = exp.obtain_rabi_params(
    targets=Q0,
    time_range=np.arange(16, 321, 16),
    n_shots=512,
    plot=True,
)

## 3. Calibrate the default half-pi and pi pulses

These are the standard pulses used by many characterization and control
routines.

In [ ]:
hpi_result = exp.calibrate_hpi_pulse(
    targets=Q0,
    n_points=21,
    n_rotations=8,
    n_shots=512,
    plot=True,
)

pi_result = exp.calibrate_pi_pulse(
    targets=Q0,
    n_points=21,
    n_rotations=8,
    n_shots=512,
    plot=True,
)

## 4. Inspect and validate the cached pulses

Plot the cached waveforms and run `repeat_sequence()` as a quick
sanity check.

In [ ]:
exp.hpi_pulse[Q0].plot()
exp.pi_pulse[Q0].plot()

repeat_hpi = exp.repeat_sequence({Q0: exp.hpi_pulse[Q0]})
repeat_pi = exp.repeat_sequence({Q0: exp.pi_pulse[Q0]})

## 5. Refine the DRAG pulses

DRAG calibration is useful once the default pulses are already close and
you want lower leakage and better benchmarking performance.

In [ ]:
drag_hpi_result = exp.calibrate_drag_hpi_pulse(
    targets=Q0,
    n_iterations=2,
    n_shots=512,
    plot=True,
)

drag_pi_result = exp.calibrate_drag_pi_pulse(
    targets=Q0,
    n_iterations=2,
    n_shots=512,
    plot=True,
)

## 6. Inspect the DRAG pulses and optionally save the note

In [ ]:
exp.drag_hpi_pulse[Q0].plot()
exp.drag_pi_pulse[Q0].plot()

repeat_drag_hpi = exp.repeat_sequence({Q0: exp.drag_hpi_pulse[Q0]})
repeat_drag_pi = exp.repeat_sequence({Q0: exp.drag_pi_pulse[Q0]})

# exp.calib_note.save()